# Capstone 2 — Data Processing and Statistical Analysis

**Aura / ClickO healthcare prep**  
Input: `NSMES1988new.csv` from Capstone 1  
Output: `NSMES1988updated.csv`


## Step 1 — Import libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

NEW_CSV = Path('NSMES1988new.csv')
print('pandas', pd.__version__)


pandas 3.0.6


## Step 2 — Load Capstone 1 output

In [2]:
df = pd.read_csv(NEW_CSV)
print('shape:', df.shape)
df.head()


shape: (4406, 18)


,visits,nvisits,ovisits,novisits,emergency,hospital,health,chronic,adl,region,age,gender,married,school,income,employed,insurance,medicaid
0,5,0,0,0,0,1,average,2,normal,other,6.9,male,yes,6,2.8810,yes,yes,no
1,1,0,2,0,2,0,average,2,normal,other,7.4,female,yes,10,2.7478,no,yes,no
2,13,0,0,0,3,3,poor,4,limited,other,6.6,female,no,10,0.6532,no,no,yes
3,16,0,5,0,1,1,poor,2,limited,other,7.6,male,yes,3,0.6588,no,yes,no
4,3,0,0,0,0,0,average,2,limited,other,7.9,female,yes,6,0.6588,no,yes,no


## Step 3 — Memory vs Capstone 1 style frame

In [3]:
# Capstone 1 deep memory was measured on the cleaned frame before/after export.
# Recompute here for the loaded new CSV and comment.
mem_new = df.memory_usage(deep=True).sum()
print('NSMES1988new deep memory bytes:', int(mem_new))
print('NSMES1988new deep memory MB:', round(mem_new / 1e6, 3))
print()
print('Comment: loading from CSV rebuilds object/string columns, so memory is similar')
print('to Capstone 1 after clean export. Category dtypes (recommended earlier) would shrink this.')


NSMES1988new deep memory bytes: 2228671
NSMES1988new deep memory MB: 2.229

Comment: loading from CSV rebuilds object/string columns, so memory is similar
to Capstone 1 after clean export. Category dtypes (recommended earlier) would shrink this.


## Step 4 — Rescale age and income

In [4]:
# age was years/10 → multiply by 10 for years
# income was USD/10000 → multiply by 10000 for USD
df['age'] = df['age'] * 10
df['income'] = df['income'] * 10000
print('age sample:', df['age'].head(3).tolist())
print('income sample:', df['income'].head(3).tolist())
df[['age', 'income']].describe()


age sample: [69.0, 74.0, 66.0]
income sample: [28809.999999999996, 27477.999999999996, 6532.0]


,age,income
count,4406.000000,4406.000000
mean,74.024058,25271.320468
std,6.334050,29246.476178
min,66.000000,-10125.000000
25%,69.000000,9121.500000
50%,73.000000,16981.500000
75%,78.000000,31728.500000
max,109.000000,548351.000000


## Step 5 — Basic statistical analysis (manual) + save updated CSV

In [5]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print('numeric columns:', num_cols)

manual = pd.DataFrame({
    'count': df[num_cols].count(),
    'mean': df[num_cols].mean(),
    'std': df[num_cols].std(),
    'min': df[num_cols].min(),
    '25%': df[num_cols].quantile(0.25),
    '50%': df[num_cols].median(),
    '75%': df[num_cols].quantile(0.75),
    'max': df[num_cols].max(),
})
print(manual.round(3))

out = Path('NSMES1988updated.csv')
df.to_csv(out, index=False)
print('wrote', out)


numeric columns: ['visits', 'nvisits', 'ovisits', 'novisits', 'emergency', 'hospital', 'chronic', 'age', 'school', 'income']
           count       mean        std      min     25%      50%      75%  \
visits      4406      5.774      6.759      0.0     1.0      4.0      8.0   
nvisits     4406      1.618      5.317      0.0     0.0      0.0      1.0   
ovisits     4406      0.751      3.653      0.0     0.0      0.0      0.0   
novisits    4406      0.536      3.880      0.0     0.0      0.0      0.0   
emergency   4406      0.264      0.704      0.0     0.0      0.0      0.0   
hospital    4406      0.296      0.746      0.0     0.0      0.0      0.0   
chronic     4406      1.542      1.350      0.0     1.0      1.0      2.0   
age         4406     74.024      6.334     66.0    69.0     73.0     78.0   
school      4406     10.290      3.739      0.0     8.0     11.0     12.0   
income      4406  25271.320  29246.476 -10125.0  9121.5  16981.5  31728.5   

                max  
visit

### Brief outcome report (manual stats)
- After rescaling, age reads as years and income as USD — ranges should look realistic vs Capstone 1.
- Visit-related columns are skewed (many zeros / low counts with some higher values) — typical for healthcare utilization.
- `school` (years of education) and `chronic` are small integers; means are interpretable.
- Non-numeric factor columns are excluded from this manual numeric table on purpose.


## Step 6 — compare `describe()` vs manual stats

In [6]:
desc = df.describe(include='all')
print(desc)
print()
# numeric-only describe should align with manual table
desc_num = df[num_cols].describe().T
compare = desc_num[['count','mean','std','min','25%','50%','75%','max']].round(3)
print('describe() numeric:')
print(compare)
print()
print('manual vs describe mean abs diff:')
print((compare['mean'] - manual['mean'].round(3)).abs())


             visits      nvisits      ovisits     novisits    emergency  \
count   4406.000000  4406.000000  4406.000000  4406.000000  4406.000000   
unique          NaN          NaN          NaN          NaN          NaN   
top             NaN          NaN          NaN          NaN          NaN   
freq            NaN          NaN          NaN          NaN          NaN   
mean       5.774399     1.618021     0.750794     0.536087     0.263504   
std        6.759225     5.317056     3.652759     3.879506     0.703659   
min        0.000000     0.000000     0.000000     0.000000     0.000000   
25%        1.000000     0.000000     0.000000     0.000000     0.000000   
50%        4.000000     0.000000     0.000000     0.000000     0.000000   
75%        8.000000     1.000000     0.000000     0.000000     0.000000   
max       89.000000   104.000000   141.000000   155.000000    12.000000   

           hospital   health      chronic     adl region          age  gender  \
count   4406.00000

## Step 7 — Columns not eligible for numeric stats + dtype notes

In [7]:
non_num = df.select_dtypes(exclude=[np.number]).columns.tolist()
print('Not eligible for numeric mean/std style stats:', non_num)
print()
print('Suggested dtype changes:')
for c in non_num:
    print(f'  - {c}: object/string → category')
print('  - count-like ints already numeric; could downcast to int16/int32')
print('  - age/income now float after rescale; float32 optional')


Not eligible for numeric mean/std style stats: ['health', 'adl', 'region', 'gender', 'married', 'employed', 'insurance', 'medicaid']

Suggested dtype changes:
  - health: object/string → category
  - adl: object/string → category
  - region: object/string → category
  - gender: object/string → category
  - married: object/string → category
  - employed: object/string → category
  - insurance: object/string → category
  - medicaid: object/string → category
  - count-like ints already numeric; could downcast to int16/int32
  - age/income now float after rescale; float32 optional


## Step 8 — Optional: apply category dtypes and export another CSV

In [8]:
df_opt = df.copy()
for c in ['health', 'adl', 'region', 'gender', 'married', 'employed', 'insurance', 'medicaid']:
    if c in df_opt.columns:
        df_opt[c] = df_opt[c].astype('category')

opt_path = Path('NSMES1988updated_optimized.csv')
df_opt.to_csv(opt_path, index=False)
print('wrote optional', opt_path)
print('memory before (deep):', int(df.memory_usage(deep=True).sum()))
print('memory after categories (deep):', int(df_opt.memory_usage(deep=True).sum()))


wrote optional NSMES1988updated_optimized.csv
memory before (deep): 2228671
memory after categories (deep): 388879


## Step 9 — Capstone 2 summary

- Loaded `NSMES1988new.csv` from Capstone 1.
- Compared memory: similar to post-clean Capstone 1 export; categories still recommended.
- Scaled `age * 10` and `income * 10000`.
- Ran manual numeric stats and compared to `describe()` — they match on numeric columns.
- Factor columns are not for mean/std until encoded; convert to category for efficiency.
- Saved `NSMES1988updated.csv` for Capstone 3/4.
